# LLM X-Ray quickstart

Measure what a local language model does inside, from Python. Everything runs on this machine.

Set `XRAY_EXAMPLE_MODEL` to a model folder or a repo id that is already in your Hugging Face cache. Nothing is downloaded.

In [ ]:
import os

import xray

m = xray.Model(os.environ.get("XRAY_EXAMPLE_MODEL", "Qwen/Qwen2.5-1.5B"))
m

## Anatomy and hardware fit

Read from the file headers, so this is instant and loads no weights.

In [ ]:
anatomy = m.inspect()
arch = anatomy["architecture"]
print(f"{arch['n_layers']} layers, {anatomy['totals']['params'] / 1e9:.2f}B parameters")
for part, n in sorted(anatomy["by_component"].items(), key=lambda kv: -kv[1])[:3]:
    print(f"  {part}: {n / anatomy['totals']['params']:.0%}")

fit = m.fit()
print("Recommended for analysis:", fit["recommended"]["analysis"])

## Timing

Runs in a separate worker process and is recorded. `Run.result` holds the full data; `Run.summary()` the headline numbers.

In [ ]:
timing = m.timing(contexts=[int(os.environ.get("XRAY_EXAMPLE_CONTEXT", "128"))])
decode = timing.result["contexts"][0]["decode"]
print(f"decode: {decode['tokens_per_s']:.1f} tokens/s")
print({k: f"{v:.0%}" for k, v in decode["shares"].items() if v > 0.02})

## Activations

Which neurons fire for which kind of prompt, and what happens when a layer is skipped.

In [ ]:
profile = m.profile(sets=["coding", "math"], limit=3, new_tokens=4, ablation_prompts=2)
for name, cmp in profile.result["comparison"].items():
    top = cmp["top_specific"][0]
    print(f"{name}: most specific neuron is layer {top['layer']}, neuron {top['neuron']}")
print("least harmful layers to skip:", profile.result["redundancy"]["most_removable"])

## Drill down, edit, compare

Queries read saved runs. An edit is applied to a copy and scored against the untouched base.

In [ ]:
print(m.query("layer 1"))

layer = profile.result["redundancy"]["most_removable"][0]
a = m.edit([f"drop_layers layers={layer}"])
b = m.edit(["prune_neurons keep=0.9"])
for row in xray.compare(a, b):
    if row["verdict"] in ("better", "worse"):
        print(f"{row['label']}: {row['a']:.3f} -> {row['b']:.3f} ({row['verdict']})")

## Report

Everything measured so far, condensed to a token budget. The text has no file paths or prompts.

In [ ]:
report = m.report(budget_tokens=3000)
print(report["text"][:600])
print(f"...\n{report['tokens_estimate']} tokens (estimate), fits budget: {report['fits']}")

## Earlier runs

Every run is kept and can be reloaded by id.

In [ ]:
for row in m.runs()[:5]:
    print(row["id"], row["type"], row["status"])
again = xray.load_run(timing.id)
assert again.result == timing.result